In [59]:
!wget http://nlp.stanford.edu/data/glove.6B.zip
!unzip glove.6B.zip

--2025-03-03 23:55:13--  http://nlp.stanford.edu/data/glove.6B.zip
Resolving nlp.stanford.edu (nlp.stanford.edu)... 171.64.67.140
Connecting to nlp.stanford.edu (nlp.stanford.edu)|171.64.67.140|:80... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://nlp.stanford.edu/data/glove.6B.zip [following]
--2025-03-03 23:55:13--  https://nlp.stanford.edu/data/glove.6B.zip
Connecting to nlp.stanford.edu (nlp.stanford.edu)|171.64.67.140|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip [following]
--2025-03-03 23:55:13--  https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip
Resolving downloads.cs.stanford.edu (downloads.cs.stanford.edu)... 171.64.64.22
Connecting to downloads.cs.stanford.edu (downloads.cs.stanford.edu)|171.64.64.22|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 862182613 (822M) [application/zip]
Saving to: ‘glove.6B.zip.2’

gl

In [61]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional, GlobalAveragePooling1D
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.regularizers import l2
import tensorflow_datasets as tfds
import numpy as np
from tensorflow.keras.layers import BatchNormalization
# Load AG News dataset
dataset, info = tfds.load('ag_news_subset', with_info=True, as_supervised=True)
train_data, test_data = dataset['train'], dataset['test']

# Prepare the data
vocab_size = 20000  # Limit vocabulary to top 20,000 words
max_length = 200    # Maximum text length (truncation/padding)

# Tokenizer to convert text to sequences
tokenizer = keras.preprocessing.text.Tokenizer(num_words=vocab_size)
tokenizer.fit_on_texts([text.numpy().decode('utf-8') for text, _ in train_data])

# Convert text data to sequences and pad them
def preprocess_dataset(dataset):
    texts, labels = [], []
    for text, label in dataset:
        texts.append(text.numpy().decode('utf-8'))
        labels.append(label.numpy())
    sequences = tokenizer.texts_to_sequences(texts)
    padded_sequences = pad_sequences(sequences, maxlen=max_length, padding='post', truncating='post')
    return padded_sequences, np.array(labels)

# Preprocess training and testing data
tr_x, tr_y = preprocess_dataset(train_data)
te_x, te_y = preprocess_dataset(test_data)

# Load GloVe 6B embeddings
glove_path = "/content/glove.6B.100d.txt"  # Update with the correct path
embedding_dim = 100  # Choose dimension (50d, 100d, 200d, or 300d)

# Create a dictionary mapping words to vectors
embeddings_index = {}
with open(glove_path, "r", encoding="utf-8") as f:
    for line in f:
        values = line.split()
        word = values[0]
        vector = np.asarray(values[1:], dtype="float32")
        embeddings_index[word] = vector

# Create an embedding matrix for the tokenizer’s vocabulary
embedding_matrix = np.zeros((vocab_size, embedding_dim))
for word, i in tokenizer.word_index.items():
    if i < vocab_size:
        embedding_vector = embeddings_index.get(word)
        if embedding_vector is not None:
            embedding_matrix[i] = embedding_vector


model = keras.Sequential([
    Embedding(vocab_size, embedding_dim, input_length=max_length, weights=[embedding_matrix], trainable=True),
    Bidirectional(LSTM(32, return_sequences=True, kernel_regularizer=l2(0.01))),
    Dropout(0.5),
    BatchNormalization(),
    Bidirectional(LSTM(16, return_sequences=False)),  # Second LSTM layer
    Dropout(0.5),
    Dense(32, activation='relu', kernel_regularizer=l2(0.01)),
    BatchNormalization(),
    Dropout(0.5),
    Dense(4, activation='softmax')
])

# Compile the model
model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

# Add early stopping
early_stopping = keras.callbacks.EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)

# Train the model
model.fit(tr_x, tr_y, epochs=20, batch_size=64, validation_data=(te_x, te_y), callbacks=[early_stopping])

# Evaluate the model
test_loss, test_acc = model.evaluate(te_x, te_y)
print(f"Test Accuracy: {test_acc:.4f}")

Epoch 1/20


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


1875/1875 ━━━━━━━━━━━━━━━━━━━━ 64s 31ms/step - accuracy: 0.6581 - loss: 1.8737 - val_accuracy: 0.8804 - val_loss: 0.4655
Epoch 2/20
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 58s 31ms/step - accuracy: 0.8865 - loss: 0.4580 - val_accuracy: 0.8959 - val_loss: 0.3598
Epoch 3/20
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 81s 31ms/step - accuracy: 0.9084 - loss: 0.3703 - val_accuracy: 0.9061 - val_loss: 0.3655
Epoch 4/20
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 83s 31ms/step - accuracy: 0.9188 - loss: 0.3293 - val_accuracy: 0.9039 - val_loss: 0.3397
Epoch 5/20
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 82s 31ms/step - accuracy: 0.9245 - loss: 0.3036 - val_accuracy: 0.9059 - val_loss: 0.3675
Epoch 6/20
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 58s 31ms/step - accuracy: 0.9324 - loss: 0.2803 - val_accuracy: 0.8933 - val_loss: 0.4177
238/238 ━━━━━━━━━━━━━━━━━━━━ 3s 13ms/step - accuracy: 0.9097 - loss: 0.3393
Test Accuracy: 0.9039


In [62]:
def predict_text_class(text, model, tokenizer, max_length):
    # Convert text to sequence
    sequence = tokenizer.texts_to_sequences([text])
    padded_sequence = pad_sequences(sequence, maxlen=max_length, padding='post', truncating='post')

    # Get prediction probabilities
    prediction = model.predict(padded_sequence)[0]  # Get the first (only) prediction

    # Print class probabilities
    for i, prob in enumerate(prediction):
        print(f"Class {i}: {prob:.4f}")

    # Predicted class
    predicted_class = np.argmax(prediction)
    print(f"Predicted Class: {predicted_class}")

    return predicted_class, prediction

In [63]:
example_sentences = [
    "The stock market is showing signs of recovery after a major downturn.",
    "The new smartphone features an advanced AI-powered camera.",
    "The local football team won their championship match last night.",
    "Researchers discovered a new method for faster data processing in quantum computing.",
    "The United Nations held an emergency meeting to address the escalating tensions between the two neighboring countries."
]

for sentence in example_sentences:
    print(f"\nSentence: {sentence}")
    predict_text_class(sentence, model, tokenizer, max_length)


Sentence: The stock market is showing signs of recovery after a major downturn.
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 358ms/step
Class 0: 0.0210
Class 1: 0.0005
Class 2: 0.9573
Class 3: 0.0213
Predicted Class: 2

Sentence: The new smartphone features an advanced AI-powered camera.
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
Class 0: 0.0019
Class 1: 0.0001
Class 2: 0.0129
Class 3: 0.9851
Predicted Class: 3

Sentence: The local football team won their championship match last night.
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
Class 0: 0.0125
Class 1: 0.9870
Class 2: 0.0002
Class 3: 0.0003
Predicted Class: 1

Sentence: Researchers discovered a new method for faster data processing in quantum computing.
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
Class 0: 0.0021
Class 1: 0.0002
Class 2: 0.0161
Class 3: 0.9817
Predicted Class: 3

Sentence: The United Nations held an emergency meeting to address the escalating tensions between the two neighboring countries.
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
Class 0: 0.9820
Class 1:

In [64]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)              │ (64, 200, 100)              │       2,000,000 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ bidirectional_2 (Bidirectional)      │ (64, 200, 64)               │          34,048 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_3 (Dropout)                  │ (64, 200, 64)               │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_2                │ (64, 200, 64)               │             256 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ bidirectional_3 (Bidirectional)      │ (64, 32)                    │          10,368 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_4 (Dropout)                  │ (64, 32)                    │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ (64, 32)                    │           1,056 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_3                │ (64, 32)                    │             128 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_5 (Dropout)                  │ (64, 32)                    │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_3 (Dense)                      │ (64, 4)                     │             132 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 6,137,582 (23.41 MB)

 Trainable params: 2,045,796 (7.80 MB)

 Non-trainable params: 192 (768.00 B)

 Optimizer params: 4,091,594 (15.61 MB)

In [107]:
import tensorflow as tf
import numpy as np
from scipy.spatial.distance import cosine

def find_closest_word(embedding_vector, tokenizer, embeddings_index):
    """
    Finds the closest word in the vocabulary to the given embedding vector.
    Ensures that the word exists in both embeddings_index and tokenizer.word_index.
    """
    min_distance = float("inf")
    closest_word = None

    for word, index in tokenizer.word_index.items():
        if word in embeddings_index:  # Ensure word has an embedding
            word_embedding = embeddings_index[word]
            distance = cosine(word_embedding, embedding_vector)
            if distance < min_distance:
                min_distance = distance
                closest_word = word

    if closest_word is None:
        print("Error: No valid word found! Choosing a random word from vocabulary.")
        closest_word = np.random.choice(list(tokenizer.word_index.keys()))

    return closest_word


def generate_single_word_uap(model, tokenizer, embeddings_index, vocab_size, embedding_dim, max_length, alpha=1, num_iterations=10):
    """
    Generates a Universal Adversarial Perturbation (UAP) consisting of a **single** adversarial word.
    """
    # Ensure embedding layer is trainable
    model.layers[0].trainable = True
    embedding_layer = model.layers[0]

    # Pick a valid random word from the tokenizer's vocabulary that exists in embeddings_index
    vocab_words = [word for word in tokenizer.word_index.keys() if word in embeddings_index]
    initial_word = np.random.choice(vocab_words)  # Now guaranteed to be in vocab
    adversarial_word = initial_word

    print(f"Starting UAP with word: {adversarial_word}")

    for _ in range(num_iterations):
        # Convert word to token index sequence
        adversarial_index = tokenizer.texts_to_sequences([[adversarial_word]])

        if not adversarial_index or len(adversarial_index[0]) == 0:
            print(f"Warning: '{adversarial_word}' not found in vocabulary. Choosing a new word.")
            adversarial_word = np.random.choice(vocab_words)  # Pick a valid word
            continue  # Restart loop with new word

        adversarial_index = tf.keras.preprocessing.sequence.pad_sequences(adversarial_index, maxlen=1, padding="post")
        adversarial_index = tf.convert_to_tensor(adversarial_index, dtype=tf.int32)

        # Get embedding for the adversarial word
        adversarial_embedding = embedding_layer(adversarial_index)  # Shape: (1, 1, embedding_dim)

        with tf.GradientTape() as tape:
            tape.watch(adversarial_embedding)

            # Convert adversarial embedding back to token index sequence
            adversarial_token = tokenizer.texts_to_sequences([[adversarial_word]])
            adversarial_token = tf.keras.preprocessing.sequence.pad_sequences(adversarial_token, maxlen=max_length, padding="post")
            adversarial_token = tf.convert_to_tensor(adversarial_token, dtype=tf.int32)

            # Forward pass through the model
            pred = model(adversarial_token)
            target_label = tf.argmax(pred, axis=1)
            loss = tf.keras.losses.sparse_categorical_crossentropy(target_label, pred)

        # Compute gradient w.r.t. the embedding
        grads = tape.gradient(loss, adversarial_embedding)

        if grads is None:
            raise ValueError("Gradient computation failed. Ensure the model and embeddings are trainable.")

        # Update the adversarial embedding
        updated_embedding = adversarial_embedding + alpha * tf.stop_gradient(grads)

        # Find the closest word in the vocabulary
        closest_word = find_closest_word(updated_embedding.numpy()[0][0], tokenizer, embeddings_index)

        # Ensure the selected word is in tokenizer's vocabulary
        if closest_word in tokenizer.word_index:
            adversarial_word = closest_word  # Update to new adversarial word
            print(f"Iteration {_+1}: Updated UAP word -> {adversarial_word}")
        else:
            print(f"Warning: Selected word '{closest_word}' is not in vocabulary, keeping previous word.")

    print(f"Final UAP Word: {adversarial_word}")
    return adversarial_word

# Generate a single-word UAP
uap_word = generate_single_word_uap(model, tokenizer, embeddings_index, vocab_size=20000, embedding_dim=100, max_length=200)
print("Generated UAP Word:", uap_word)


Starting UAP with word: jaine


ValueError: Gradient computation failed. Ensure the model and embeddings are trainable.

In [94]:
def test_uap_attack(model, tokenizer, uap_word, test_texts, true_labels):
    """
    Tests the effectiveness of the UAP word on a set of test cases.

    Args:
    - model: The trained text classification model.
    - tokenizer: Tokenizer used for text preprocessing.
    - uap_word: The adversarial word generated from the attack.
    - test_texts: List of test sentences.
    - true_labels: List of true labels for the test sentences.

    Returns:
    - attack_success_rate: Percentage of cases where the label was flipped.
    """
    successful_attacks = 0
    print(f"Best adversarial word: '{uap_word}'\n")

    original_preds = []
    adversarial_preds = []

    for i, text in enumerate(test_texts):
        # Get original prediction
        original_seq = tokenizer.texts_to_sequences([text])
        original_seq = tf.keras.preprocessing.sequence.pad_sequences(original_seq, maxlen=200, padding="post")
        original_pred = np.argmax(model.predict(original_seq), axis=1)[0]
        original_preds.append(original_pred)

        # Insert adversarial word at the beginning
        adversarial_text = uap_word + " " + text
        adversarial_seq = tokenizer.texts_to_sequences([adversarial_text])
        adversarial_seq = tf.keras.preprocessing.sequence.pad_sequences(adversarial_seq, maxlen=200, padding="post")
        adversarial_pred = np.argmax(model.predict(adversarial_seq), axis=1)[0]
        adversarial_preds.append(adversarial_pred)

        # Check if attack was successful (label flip)
        attack_success = original_pred != adversarial_pred
        if attack_success:
            successful_attacks += 1

        # Print formatted results
        print("------------------------------------------------------------")
        print(f"Original text: '{text}'")
        print(f"Original prediction: {original_pred}")
        print(f"Adversarial text: '{adversarial_text}'")
        print(f"Adversarial prediction: {adversarial_pred}")
        print(f"Attack successful: {attack_success}")
        print("------------------------------------------------------------\n")

    attack_success_rate = successful_attacks / len(test_texts)
    print(f"Original accuracy: {np.mean(np.array(original_preds) == np.array(true_labels)):.4f}")
    print(f"Adversarial accuracy: {np.mean(np.array(adversarial_preds) == np.array(true_labels)):.4f}")
    print(f"Attack success rate: {attack_success_rate:.4f}")

    return attack_success_rate

# Example test case execution
test_sentences = [
    "The economy is showing signs of recovery according to latest reports",
    "Scientists discover new species in Amazon rainforest",
    "Sports team wins championship for the third time"
]

true_labels = [0, 3, 1]  # Example true labels for test cases
test_uap_attack(model, tokenizer, uap_word, test_sentences, true_labels)


Best adversarial word: 'overtook'

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 74ms/step
------------------------------------------------------------
Original text: 'The economy is showing signs of recovery according to latest reports'
Original prediction: 2
Adversarial text: 'overtook The economy is showing signs of recovery according to latest reports'
Adversarial prediction: 3
Attack successful: True
------------------------------------------------------------

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 107ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
------------------------------------------------------------
Original text: 'Scientists discover new species in Amazon rainforest'
Original prediction: 3
Adversarial text: 'overtook Scientists discover new species in Amazon rainforest'
Adversarial prediction: 3
Attack successful: False
------------------------------------------------------------

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
-------

0.3333333333333333

In [108]:
import tensorflow as tf
import numpy as np
from tqdm import tqdm
from sklearn.metrics import accuracy_score
import random

def implement_uap_attack(model, tokenizer, train_data, test_data, vocab_size=500, max_length=200,
                         num_iterations=5, learning_rate=1.0, batch_size=8, insertion_position=0):
    """
    Implements a Universal Adversarial Perturbation (UAP) attack for text classifiers using gradients.

    Args:
        model: The target model to attack
        tokenizer: Tokenizer used by the model
        train_data: Training data as (texts, labels) for finding the adversarial word
        test_data: Test data as (texts, labels) for evaluation
        vocab_size: Size of the vocabulary
        max_length: Maximum sequence length
        num_iterations: Number of iterations for optimizing the adversarial word
        learning_rate: Learning rate for gradient updates
        batch_size: Batch size for training
        insertion_position: Position to insert the adversarial word (0 = beginning)

    Returns:
        adversarial_word: The found adversarial word
        original_accuracy: Model accuracy without the attack
        adversarial_accuracy: Model accuracy under attack
    """
    # Get embedding layer from the model
    embedding_layer = model.layers[0]
    embedding_weights = embedding_layer.get_weights()[0]
    embedding_dim = embedding_weights.shape[1]

    # Initialize adversarial word with a random word from vocabulary
    random_word_idx = random.randint(1, vocab_size - 1)  # Skip 0 (padding)
    adv_embedding = embedding_weights[random_word_idx].copy()

    # Create a mapping from embedding to word (inverse of tokenizer)
    word_to_idx = tokenizer.word_index
    idx_to_word = {v: k for k, v in word_to_idx.items()}

    # Get a subset of training data for optimization
    train_texts, train_labels = train_data

    # Evaluate original accuracy
    test_texts, test_labels = test_data
    original_preds = model.predict(test_texts)
    original_accuracy = accuracy_score(test_labels, np.argmax(original_preds, axis=1))
    print(f"Original model accuracy: {original_accuracy:.4f}")

    # Optimization loop
    best_adv_word = None
    best_attack_success_rate = 0

    for iteration in tqdm(range(num_iterations)):
        # Choose a random batch from training data
        batch_indices = np.random.choice(len(train_labels), batch_size, replace=False)
        batch_x = train_texts[batch_indices]
        batch_y = train_labels[batch_indices]

        # Create a TensorFlow variable for the adversarial embedding
        adv_embedding_var = tf.Variable(adv_embedding, dtype=tf.float32)

        # Compute gradients
        with tf.GradientTape() as tape:
            # Create adversarial examples by inserting the word
            adv_inputs = np.copy(batch_x)
            for i in range(len(adv_inputs)):
                adv_inputs[i, insertion_position] = random_word_idx

            # Compute model predictions
            embedded_inputs = embedding_layer(adv_inputs)
            predictions = model(embedded_inputs)

            # Maximize loss for non-targeted attack
            loss = -tf.reduce_mean(tf.keras.losses.sparse_categorical_crossentropy(batch_y, predictions))

        # Compute gradients with respect to embedding
        grads = tape.gradient(loss, adv_embedding_var)

        # Update adversarial embedding (gradient ascent for non-targeted attack)
        adv_embedding = adv_embedding_var.numpy() + learning_rate * grads.numpy()

        # Project back to vocabulary space (find the closest word)
        similarities = np.dot(embedding_weights[1:vocab_size], adv_embedding)  # Skip padding token
        closest_word_idx = np.argmax(similarities) + 1  # Offset by 1 to account for skipping 0
        adv_embedding = embedding_weights[closest_word_idx].copy()

        # Evaluate attack success rate periodically
        if iteration % 2 == 0 or iteration == num_iterations - 1:
            # Apply the adversarial word to test data
            test_adv_texts = np.copy(test_texts)
            for i in range(len(test_adv_texts)):
                test_adv_texts[i, insertion_position] = closest_word_idx

            adv_preds = model.predict(test_adv_texts)
            adv_accuracy = accuracy_score(test_labels, np.argmax(adv_preds, axis=1))
            attack_success_rate = original_accuracy - adv_accuracy

            print(f"Iteration {iteration}: Adversarial word = '{idx_to_word.get(closest_word_idx, '<UNK>')}', "
                  f"Attack success rate = {attack_success_rate:.4f}")

            if attack_success_rate > best_attack_success_rate:
                best_attack_success_rate = attack_success_rate
                best_adv_word = idx_to_word.get(closest_word_idx, '<UNK>')

    # Final evaluation
    print(f"\nBest adversarial word: '{best_adv_word}'")
    print(f"Original accuracy: {original_accuracy:.4f}")
    print(f"Adversarial accuracy: {original_accuracy - best_attack_success_rate:.4f}")
    print(f"Attack success rate: {best_attack_success_rate:.4f}")

    return best_adv_word, original_accuracy, original_accuracy - best_attack_success_rate

In [109]:
def run_uap_attack(model, tokenizer):
    # Get data
    tr_x, tr_y = preprocess_dataset(train_data)
    te_x, te_y = preprocess_dataset(test_data)

    print("Running UAP attack...")
    # Try the complex implementation first
    try:
        adv_word, orig_acc, adv_acc = implement_uap_attack(
            model=model,
            tokenizer=tokenizer,
            train_data=(tr_x, tr_y),
            test_data=(te_x, te_y),
            vocab_size=500,  # <-- Reduced vocab_size
            max_length=max_length,
            num_iterations=5,  # <-- Reduced num_iterations
            learning_rate=1.0,
            batch_size=8,  # <-- Reduced batch_size
            insertion_position=0
        )
    except Exception as e:
        print(f"Complex implementation failed with error: {e}")
        print("Falling back to simple implementation...")
        adv_word, orig_acc, adv_acc = implement_uap_attack_simple(
            model=model,
            tokenizer=tokenizer,
            train_data=(tr_x, tr_y),
            test_data=(te_x, te_y),
            vocab_size=500,  # <-- Reduced vocab_size
            max_length=max_length,
            insertion_position=0
        )

    # Test the adversarial word on specific examples
    def test_example(text):
        # Process the original text
        sequence = tokenizer.texts_to_sequences([text])[0]
        padded_sequence = pad_sequences([sequence], maxlen=max_length, padding='post', truncating='post')
        original_pred = np.argmax(model.predict(padded_sequence, verbose=0)[0])

        # Process the text with adversarial word inserted
        adv_word_id = tokenizer.word_index.get(adv_word, 0)
        adv_sequence = [adv_word_id] + sequence
        adv_padded_sequence = pad_sequences([adv_sequence], maxlen=max_length, padding='post', truncating='post')
        adv_pred = np.argmax(model.predict(adv_padded_sequence, verbose=0)[0])

        print(f"Original text: '{text}'")
        print(f"Original prediction: {original_pred}")
        print(f"Adversarial text: '{adv_word} {text}'")
        print(f"Adversarial prediction: {adv_pred}")
        print(f"Attack successful: {original_pred != adv_pred}")
        print("-" * 50)

    # Test on some examples
    examples = [
        "The economy is showing signs of recovery according to latest reports",
        "Scientists discover new species in Amazon rainforest",
        "Sports team wins championship for the third time"
    ]

    for example in examples:
        test_example(example)

    return adv_word, orig_acc, adv_acc

# Call this function to run the attack
adv_word, orig_acc, adv_acc = run_uap_attack(model, tokenizer)

Running UAP attack...
238/238 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step
Original model accuracy: 0.9039


  0%|          | 0/5 [00:00<?, ?it/s]

Complex implementation failed with error: Exception encountered when calling Sequential.call().

Invalid input shape for input [[[-0.3450698   0.18223047  0.32903343 ... -0.36199546  0.43335792
    0.15132041]
  [-0.80127066  0.43610233  0.33131647 ...  0.90995437  1.9077895
   -0.27474773]
  [ 0.25661415  0.66994464 -0.23616587 ... -0.30285215  0.65126187
   -0.10727379]
  ...
  [-0.0389509   0.00600201  0.11154952 ... -0.1187248   0.36532527
    0.04451871]
  [-0.0389509   0.00600201  0.11154952 ... -0.1187248   0.36532527
    0.04451871]
  [-0.0389509   0.00600201  0.11154952 ... -0.1187248   0.36532527
    0.04451871]]

 [[-0.3450698   0.18223047  0.32903343 ... -0.36199546  0.43335792
    0.15132041]
  [-0.00517911  0.44313243 -0.638871   ... -0.59031296  0.7498003
    0.01235767]
  [ 0.2576603  -0.38994148 -0.0451759  ... -0.38672993  0.75289714
    0.2027804 ]
  ...
  [-0.0389509   0.00600201  0.11154952 ... -0.1187248   0.36532527
    0.04451871]
  [-0.0389509   0.00600201  0.1

238/238 ━━━━━━━━━━━━━━━━━━━━ 3s 12ms/step
Original model accuracy: 0.9039


  0%|          | 1/499 [00:05<42:51,  5.16s/it]

...

New best word: 'software', Attack success rate: 0.0764


 33%|███▎      | 165/499 [10:55<23:44,  4.27s/it]

New best word: 'afp', Attack success rate: 0.1108


100%|██████████| 499/499 [31:38<00:00,  3.81s/it]



Best adversarial word: 'afp'
Original accuracy: 0.9039
Adversarial accuracy: 0.7932
Attack success rate: 0.1108
Original text: 'The economy is showing signs of recovery according to latest reports'
Original prediction: 2
Adversarial text: 'afp The economy is showing signs of recovery according to latest reports'
Adversarial prediction: 0
Attack successful: True
--------------------------------------------------
Original text: 'Scientists discover new species in Amazon rainforest'
Original prediction: 3
Adversarial text: 'afp Scientists discover new species in Amazon rainforest'
Adversarial prediction: 3
Attack successful: False
--------------------------------------------------
Original text: 'Sports team wins championship for the third time'
Original prediction: 1
Adversarial text: 'afp Sports team wins championship for the third time'
Adversarial prediction: 1
Attack successful: False
--------------------------------------------------
